In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

The function `range` takes two arguments:
- `start` is an integer,
- `stop` is an integer.

It returns the set of all integers from `start` up to and including `stop`.  `List.init k f` creates the list `[f 0; f 1; ...; f (k-1)]`.

In [ ]:
let range (start : int) (stop : int) : int RSet.t =
  RSet.of_list (List.init (stop - start + 1) (fun i -> start + i))

# The Eight Queens Puzzle

The <a href="https://en.wikipedia.org/wiki/Eight_queens_puzzle">eight queens puzzle</a> is the problem of placing eight chess queens on a chessboard so that no two queens can attack each other.  In <a href="https://en.wikipedia.org/wiki/Chess">chess</a> a queen can attack another piece if this piece is either
<ol>
    <li>in the same row,</li>
    <li>in the same column, or</li>
    <li>in the same diagonal.</li>
</ol>
The image below shows a queen in row 3, column 4.  All the locations where a piece can be captured by this queen are marked with an arrow.

<img src="queen-captures.png">

We will solve this puzzle by coding it as a formula of propositional logic.  This formula will be solvable iff the eight queens puzzle has a solution.  We will use the algorithm of *Davis and Putnam* to compute the solution of this formula.  The notebook `07-Davis-Putnam-JW.ipynb` provides the function `solve`.  It imports the notebook `04-CNF.ipynb`, which defines the types `literal`, `clause`, and `cnf`.

In [ ]:
import_notebook "07-Davis-Putnam-JW.ipynb"

The function $\texttt{var\_name}(r, c)$ takes a row $r$ and a column $c$ and returns the string $\texttt{Q<}r\texttt{,}c\texttt{>}$.  This string is interpreted as a propositional variable specifying that there is a queen in row $r$ and column $c$.  The image below shows how these variables correspond to the positions on a chess board.

<img src="queens-vars.png">

The function `var_name` takes two arguments:
- `row` is the number of a row,
- `col` is the number of a column.

It returns a string of the form `Q<row,col>`.  This string is interpreted as a propositional variable.  This variable is `true` if there is a queen in the given row and column on the board.  `Printf.sprintf` works like `Printf.printf`, but returns the formatted string instead of printing it.

In [ ]:
let var_name (row : int) (col : int) : string =
  Printf.sprintf "Q<%d,%d>" row col

In [ ]:
var_name 11 3

The function `at_most_one` takes one argument:
- `s` is a set of propositional variables.

It returns a set of clauses that expresses the fact that **at most one** of the variables in `s` is `true`:
$$ \texttt{at\_most\_one}(S) = \bigl\{ \{\neg p, \neg q\} \bigm| \langle p, q\rangle \in S \times S \wedge p \not= q\bigr\} $$

**Implementation:** For a fixed variable `p`, `RSet.filter_map` computes the clauses $\{\neg p, \neg q\}$ for all variables $q \not= p$.  `RSet.flat_map` collects these clauses for all variables `p` of `s`.

In [ ]:
let at_most_one (s : string RSet.t) : cnf =
  RSet.flat_map
    (fun p -> RSet.filter_map (fun q -> if p <> q then Some (RSet.of_list [Neg p; Neg q]) else None) s)
    s

In [ ]:
at_most_one (RSet.of_list ["a"; "b"; "c"])

The function `at_most_one_in_row` takes two arguments:
- `row` is the number of a row,
- `n` is the size of the board.

It returns a set of clauses that is `true` if and only if there is at most one queen in the given row.

In [ ]:
let at_most_one_in_row (row : int) (n : int) : cnf =
  at_most_one (RSet.map (fun col -> var_name row col) (range 1 n))

In [ ]:
RSet.map (fun col -> var_name 2 col) (range 1 4)

In [ ]:
at_most_one_in_row 2 4

The function `one_in_column` takes two arguments:
- `col` is the number of a column,
- `n` is the size of the board.

It returns a set of clauses that is true if and only if there is at least one queen in the given column.  This set contains just one clause, namely the clause consisting of the variables $\texttt{Q<}r\texttt{,}\texttt{col>}$ for all rows $r$.

In [ ]:
let one_in_column (col : int) (n : int) : cnf =
  RSet.singleton (RSet.map (fun row -> Pos (var_name row col)) (range 1 n))

In [ ]:
one_in_column 2 4

The function `at_most_one_in_falling_diagonal` takes two arguments:
- `k` is an integer that specifies a falling diagonal,
- `n` is the size of the board.

It returns a set of clauses that is `true` if and only if there is at most one queen in the falling diagonal specified by the equation
```
     row - col = k.
```
**Implementation:** `RSet.cartesian_product numbers numbers` is the set of all positions `(row, col)` on the board.  The function passed to `RSet.filter_map` destructures every position into its components `row` and `col` and keeps the positions on the diagonal.

In [ ]:
let at_most_one_in_falling_diagonal (k : int) (n : int) : cnf =
  let numbers = range 1 n in
  at_most_one
    (RSet.filter_map
       (fun (row, col) -> if row - col = k then Some (var_name row col) else None)
       (RSet.cartesian_product numbers numbers))

In [ ]:
at_most_one_in_falling_diagonal 0 4

The function `at_most_one_in_rising_diagonal` takes two arguments:
- `k` is an integer that specifies a rising diagonal,
- `n` is the size of the board.

It returns a set of clauses that is `true` if and only if there is at most one queen in the rising diagonal specified by the equation
```
    row + col = k.
```

In [ ]:
let at_most_one_in_rising_diagonal (k : int) (n : int) : cnf =
  let numbers = range 1 n in
  at_most_one
    (RSet.filter_map
       (fun (row, col) -> if row + col = k then Some (var_name row col) else None)
       (RSet.cartesian_product numbers numbers))

In [ ]:
at_most_one_in_rising_diagonal 5 4

The function `all_clauses` takes one argument:
- `n` is the size of the board.

It returns a set of clauses that specifies that
* there is at most one queen in every row,
* there is at most one queen in every rising diagonal,
* there is at most one queen in every falling diagonal, and
* there is at least one queen in every column.

**Implementation:** `RSet.flat_map` computes the clauses for every row, diagonal, or column, respectively, and `RSet.unions` combines the four sets of clauses.

In [ ]:
let all_clauses (n : int) : cnf =
  RSet.unions
    [RSet.flat_map (fun row -> at_most_one_in_row row n) (range 1 n);
     RSet.flat_map (fun k -> at_most_one_in_rising_diagonal k n) (range 3 (2 * n - 1));
     RSet.flat_map (fun k -> at_most_one_in_falling_diagonal k n) (range (2 - n) (n - 2));
     RSet.flat_map (fun col -> one_in_column col n) (range 1 n)]

The function `clause_to_string` takes one argument:
- `c` is a clause.

It converts `c` into a string.  Then we print all clauses for the eight queens puzzle.

In [ ]:
let clause_to_string (c : clause) : string =
  RSet.to_string (function Pos p -> p | Neg p -> "¬" ^ p) c

In [ ]:
RSet.iter (fun c -> print_endline (clause_to_string c)) (all_clauses 8)

The set of all clauses contains 512 clauses.  There are 64 variables.

In [ ]:
RSet.cardinal (all_clauses 8)

The function `queens` takes one argument:
- `n` is the size of the board.

It solves the *n queens problem*.  The result is the set of unit clauses computed by `solve`.  If the problem is not solvable, a message is printed and the set $\bigl\{\{\}\bigr\}$ is returned.

In [ ]:
let queens (n : int) : cnf =
  let clauses  = all_clauses n in
  let solution = solve clauses in
  if RSet.mem RSet.empty solution then
    Printf.printf "The problem is not solvable for %d queens!\n" n;
  solution

The *8 queens problem* can be solved in less than a tenth of a second using the
pure *Davis Putnam algorithm* that does not use the Jeroslow-Wang heuristic. If we want to solve the *16 queens problem*, we need to use the Jeroslow-Wang heuristic.  The function `time`, which is defined in `setup.ml`, measures the time that the computation takes.

In [ ]:
let solution = time "queens" (fun () -> queens 16)

The queens are located at the positions given by the positive literals of the solution.  The function `first` from the notebook `07-Davis-Putnam-JW.ipynb` returns the only element of a unit clause.

In [ ]:
RSet.filter_map (fun u -> match first u with Pos p -> Some p | Neg _ -> None) solution

# Visualization

The function `remove_negative_literals` takes one argument:
* `solution` is a set of unit clauses.

The function returns the set of the variables of those unit clauses in `solution` that contain a positive literal.

In [ ]:
let remove_negative_literals (solution : cnf) : string RSet.t =
  RSet.flat_map (RSet.filter_map (function Pos p -> Some p | Neg _ -> None)) solution

In [ ]:
let input_set = RSet.of_list [RSet.of_list [Pos "p"];
                              RSet.of_list [Neg "q"];
                              RSet.of_list [Neg "r"];
                              RSet.of_list [Pos "s"]];;
RSet.to_string Fun.id (remove_negative_literals input_set)

The function `extract_row_col` takes one argument:
* `var_name` is a string of the form `Q<row,col>`.

It returns the pair `(row, col)`.

**Implementation:** `String.index s c` returns the position of the first occurrence of the character `c` in the string `s`.  `String.sub s start len` returns the substring of `s` that starts at position `start` and has length `len`, and `int_of_string` converts a string of digits into an integer.

In [ ]:
let extract_row_col (var_name : string) : int * int =
  let left  = String.index var_name '<' in
  let comma = String.index var_name ',' in
  let right = String.index var_name '>' in
  let row   = String.sub var_name (left + 1) (comma - left - 1) in
  let col   = String.sub var_name (comma + 1) (right - comma - 1) in
  (int_of_string row, int_of_string col)

In [ ]:
extract_row_col "Q<13,9>"

The function `transform` takes one argument:
* `solution` is a set of unit clauses representing a solution of the *n* queens puzzle.

It returns an *association list* that maps every row to the column of the queen in this row.  An association list is a list of pairs `(key, value)`.  The function `List.assoc_opt key l` returns `Some value` if the list `l` contains a pair `(key, value)`, and `None` otherwise.

In [ ]:
let transform (solution : cnf) : (int * int) list =
  List.map extract_row_col (RSet.to_list (remove_negative_literals solution))

In order to display HTML in the notebook, we load the library `jupyter.notebook`, which is part of the <samp>OCaml</samp> kernel.  It provides the function `Jupyter_notebook.display mime data`, which displays `data` according to the MIME type `mime`.

In [ ]:
#require "jupyter.notebook";;

The function `show_solution` takes two arguments:
* `solution` is a set of unit clauses representing a solution to the *n* queens puzzle.
  The function transforms this set into an association list that maps rows to columns.
  If this list contains the pair `(row, col)`, then the queen in `row` is placed in `col`.
* `width` specifies the size of the board as a percentage of the width of the notebook.

It displays the solution on a chess board.

**Implementation:** The parameter `?(width = "50%")` is an *optional argument* with the default value `"50%"`.  When the function is called, it can be given as `~width:"70%"`, otherwise the default value is used.  The HTML code is collected in a `Buffer.t`, which is a string that can grow efficiently.  `Buffer.add_string` appends a string to the buffer and `Buffer.contents` returns the complete string.  `Jupyter_notebook.display` returns an identifier of the displayed output, which we do not need.  Therefore, it is discarded by `ignore`.

In [ ]:
let show_solution ?(width = "50%") (solution : cnf) : unit =
  let transformed = transform solution in
  let n    = List.length transformed in
  let html = Buffer.create 4096 in
  Buffer.add_string html
    (Printf.sprintf "<div style=\"display:grid; grid-template-columns:repeat(%d, 1fr); width:%s; aspect-ratio: 1/1; border: 2px solid black;\">" n width);
  for row = 1 to n do
    for col = 1 to n do
      let bg_color = if (row + col) mod 2 = 0 then "#f0d9b5" else "#b58863" in
      let cell_content =
        if List.assoc_opt row transformed = Some col then
          "<svg viewBox=\"0 0 100 100\" style=\"width: 80%; height: 80%; display: block;\">\
           <text x=\"50%\" y=\"55%\" font-size=\"90\" text-anchor=\"middle\" dominant-baseline=\"middle\" fill=\"black\">♕</text></svg>"
        else
          ""
      in
      Buffer.add_string html
        (Printf.sprintf "<div style=\"display: flex; align-items: center; justify-content: center; background-color:%s; overflow: hidden;\">%s</div>"
           bg_color cell_content)
    done
  done;
  Buffer.add_string html "</div>";
  ignore (Jupyter_notebook.display "text/html" (Buffer.contents html))

In [ ]:
show_solution ~width:"70%" solution